# Week 6 - Day 1 (Second Half)
## Task 1 - Cosine Similarity from Scratch

In [ ]:
# !pip install numpy scikit-learn sentence-transformers matplotlib
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

# Task 1 - Cosine Similarity from Scratch

In [ ]:
def cosine_similarity(a, b):
    """cos(theta) = dot(a, b) / (||a|| * ||b||)"""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        raise ValueError("Cosine similarity is undefined for a zero vector")
    return float(np.dot(a, b) / denom)

tests = [([1, 2, 3], [2, 4, 6], 1.0),
         ([1, 0], [0, 1], 0.0),
         ([1, 0], [-1, 0], -1.0)]

print("Mine vs sklearn")
for a, b, expected in tests:
    mine = cosine_similarity(a, b)
    sk = sk_cosine([a], [b])[0][0]
    print(f"{a} vs {b} -> mine={mine:.4f} | sklearn={sk:.4f} | expected={expected} | match={np.isclose(mine, sk)}")

print()
print(" | ".join(f"{a} vs {b} -> {round(cosine_similarity(a, b), 4)}" for a, b, _ in tests))

### Why cosine ignores vector length, and when that matters
`cos(a, b) = (a . b) / (||a|| ||b||)` - dividing by both norms removes magnitude, so only the **angle** between the vectors is left. `[1,2,3]` and `[2,4,6]` point the same way, so the score is 1.0 even though one is twice as long.

**When it is good:** comparing meaning/topic. A long document and a short one on the same subject get similar embeddings directions; word counts or document length shouldn't change relevance.

**When it matters (can hurt):** if magnitude carries information (e.g. rating strength, purchase counts, how *intense* a signal is), cosine throws it away. `[1,1]` and `[100,100]` look identical. In those cases use Euclidean distance or the raw dot product.

In [ ]:
# Normalised vectors: cosine == dot product
a = np.array([3.0, 4.0, 12.0])
b = np.array([1.0, 7.0, 2.0])

a_n = a / np.linalg.norm(a)
b_n = b / np.linalg.norm(b)

print("norms after normalising:", np.linalg.norm(a_n), np.linalg.norm(b_n))
print("cosine(a, b)   :", cosine_similarity(a, b))
print("dot(a_n, b_n)  :", np.dot(a_n, b_n))
print("cosine(a_n,b_n):", cosine_similarity(a_n, b_n))
print("Equal?", np.isclose(cosine_similarity(a, b), np.dot(a_n, b_n)))

Because `||a_n|| = ||b_n|| = 1`, the denominator is 1 and cosine reduces to the dot product. That is why vector databases often store normalised embeddings and use a fast dot product.